# Driver — độ bền của static Android malware detection dưới obfuscation

Notebook này chỉ **điều khiển**; mọi logic nằm trong `src/`. Chạy tuần tự theo phase.

| Phase | Cell | Thời gian | Chạy lại mỗi session? |
|---|---|---|---|
| 0 | Setup + smoke test | 30 phút | Có |
| 1 | Giải nén dataset + manifest + split | ~30 phút | Có (trừ manifest/split đã ở Drive) |
| 2 | Trích feature APK sạch | ~1,1 giờ | Không — checkpoint ở Drive |
| 3 | Baseline ML + eval sạch | ~1 giờ | Không |
| 4 | Obfuscation | 9–10 giờ, **chia 2 session** | Resume từ `obf_progress.json` |
| 5 | Ma trận kết quả | ~2 giờ | Không |

**Đừng đi tiếp nếu smoke test ở Phase 0 hỏng.**

Hai việc phải làm thủ công một lần, vì không tự động hoá được:

- **Dataset** nằm sau form đăng ký của CIC — xem Phase 1.
- **Obfuscapk** không có trên PyPI — cell ngay dưới lo việc này.

## Phase 0 — Setup

In [7]:
# Mount Drive để checkpoint
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/apk-robustness'
SCRATCH = '/content/apkrob'

import os
os.environ['APKROB_WORK'] = WORK
os.environ['APKROB_SCRATCH'] = SCRATCH
os.makedirs(WORK, exist_ok=True)
print('WORK   ', WORK)
print('SCRATCH', SCRATCH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
WORK    /content/drive/MyDrive/apk-robustness
SCRATCH /content/apkrob


In [8]:
# Java + Android build-tools cho Obfuscapk.
# apktool KHÔNG có trong danh sách gốc của PLAN nhưng Obfuscapk bắt buộc phải có —
# thiếu nó thì mọi kỹ thuật đều fail ngay ở bước decompile.
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jdk-headless apktool zipalign apksigner

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


### Obfuscapk — ba cái bẫy, không phải một

PLAN mục 2 ghi `pip install obfuscapk`. Lệnh đó không bao giờ chạy được, và vòng qua được nó rồi thì còn hai cái nữa đợi sẵn.

**1. Không có trên PyPI.** `pip install obfuscapk` trả `No matching distribution found`. Repo cũng không có `setup.py` ở gốc nên `pip install git+...` cũng hỏng. Cách duy nhất: clone rồi đưa `src/` của nó vào `PYTHONPATH` qua biến `OBFUSCAPK_SRC`.

**2. Yapsy hỏng trên Python 3.12 trở lên.** `src/requirements.txt` của Obfuscapk ghim `Yapsy==1.12.2` — bản mới nhất trên PyPI, phát hành 2019 — và bản đó `import imp`, module đã bị xoá khỏi Python 3.12. Colab hiện chạy **Python 3.13**. Nhánh master của Yapsy đã chuyển sang `importlib` nhưng chưa bao giờ được phát hành, nên phải cài từ git.

**3. BundleDecompiler.** `check_external_tool_dependencies()` khởi tạo BundleDecompiler **vô điều kiện**, dù README gọi nó là tuỳ chọn. Thiếu biến môi trường thì constructor ném `TypeError: stat: path should be string... not NoneType` — một bug của chính Obfuscapk, và vì nó chạy trước cả argparse nên ngay `--help` cũng chết.

`src/obfuscate.py` tự lo cái thứ 3: constructor chỉ gọi `os.path.isfile` chứ không chạy jar, và pipeline này chỉ truyền `.apk` chứ không bao giờ truyền `.aab`, nên một file người thế có `chmod +x` là đủ và an toàn. Nếu sau này cần xử lý `.aab` thật thì tải jar thật rồi đặt `BUNDLE_DECOMPILER_PATH` — code không ghi đè biến bạn đã đặt.

Các pin còn lại của Obfuscapk cũng từ 2021 (`pycryptodome==3.12.0`), không có wheel cho Python 3.13, nên cell dưới cài bản mới nhất thay vì theo pin.

Cell "Kiểm tra toolchain" bên dưới báo chính xác cái nào hỏng, nếu có.

In [9]:
import os, sys
print('Python', sys.version.split()[0])

# 1. Obfuscapk: clone, KHÔNG pip install (không có trên PyPI)
OBF = '/content/Obfuscapk'
if not os.path.isdir(OBF):
    !git clone -q --depth 1 https://github.com/ClaudiuGeorgiu/Obfuscapk.git {OBF}
os.environ['OBFUSCAPK_SRC'] = f'{OBF}/src'   # src/obfuscate.py đọc biến này

# 2. Phụ thuộc của Obfuscapk — bỏ pin cũ, trừ Yapsy phải lấy từ git
#    (bản trên PyPI dùng 'imp', đã bị xoá khỏi Python 3.12+)
!pip -q install pycryptodome tqdm vt-py
!pip -q install 'yapsy @ git+https://github.com/tibonihoo/yapsy.git@master#subdirectory=package'

# 3. Phụ thuộc của dự án này
!pip -q install androguard==4.1.2 scikit-learn pandas pyarrow xgboost networkx joblib

print('OBFUSCAPK_SRC =', os.environ['OBFUSCAPK_SRC'])

Python 3.13.15
  Preparing metadata (setup.py) ... done
OBFUSCAPK_SRC = /content/Obfuscapk/src


In [16]:
# Lấy code của dự án này.
REPO = 'https://github.com/trantrien1/AndroiDetection1.git'
PROJ = '/content/AndroiDetection1'

import os
if not os.path.isdir(PROJ):
    !git clone -q {REPO} {PROJ}
else:
    
    !git -C {PROJ} pull -q  
    print("Da pull")        # lấy bản mới nhất khi chạy lại session

%cd {PROJ}
if not os.path.isdir('src'):
    raise SystemExit(f'Chưa có code trong {PROJ}/src — kiểm tra lại REPO.')
print('OK:', sorted(os.listdir('src')))

Da pull
/content/AndroiDetection1
OK: ['__init__.py', '__pycache__', 'config.py', 'download.py', 'evaluate.py', 'features', 'matrix.py', 'obfuscate.py', 'split.py', 'train.py', 'utils.py']


In [11]:
# Kiểm tra toolchain TRƯỚC khi tải dataset — apktool, apksigner, zipalign và
# bản thân Obfuscapk. In ra chính xác cái nào hỏng và cách sửa.
# Phải in "THIEU / HONG: khong co - san sang" thì mới đi tiếp.
!python -m src.obfuscate --check

10:07:09 INFO    obfuscate |   apktool    /usr/bin/apktool
10:07:09 INFO    obfuscate |   apksigner  /usr/bin/apksigner
10:07:09 INFO    obfuscate |   zipalign   /usr/bin/zipalign
10:07:09 INFO    obfuscate | Tao file nguoi cho BundleDecompiler tai /content/apkrob/BundleDecompiler.jar (Obfuscapk doi no ton tai du ta chi xu ly .apk)
10:07:09 INFO    obfuscate |   obfuscapk  ok

THIEU / HONG: khong co - san sang


## Phase 1 — Lấy dataset + manifest + chốt split

**URL trong PLAN mục 3 đã chết.** `cicresearch.ca/.../Dataset/APKs/` giờ 302 về trang giới thiệu datasets, và CIC đã đặt toàn bộ dataset sau một **form đăng ký** (họ tên, email, tổ chức, chức danh, quốc gia). Không còn đường tải ẩn danh, nên notebook không thể tự tải hộ.

Tải về một lần, để trên Drive, rồi dùng lại mãi:

1. Mở [trang dataset](https://www.unb.ca/cic/datasets/maldroid-2020.html) → **Download the dataset** → điền form.
2. Tải 5 file zip theo category vào một thư mục trên Drive, ví dụ `MyDrive/maldroid_zips/`.
3. Chỉnh `ZIPS` ở cell dưới cho trỏ đúng thư mục đó.

Tên file không cần khớp chính xác — `find_local_zip` khớp không phân biệt hoa thường, miễn tên file có chứa tên category (`Benign`, `Adware`, `Banking`, `SMS`, `Riskware`).

Đặt zip trên Drive chứ đừng đặt ở `/content`: `/content` bị xoá mỗi session, còn APK giải nén thì vẫn nên để ở `/content` cho nhanh.

In [ ]:
# Trỏ tới thư mục chứa 5 file zip đã tải thủ công về Drive.
ZIPS = '/content/drive/MyDrive/maldroid_zips'

import os
os.makedirs(ZIPS, exist_ok=True)        # tạo sẵn để bạn kéo thả zip vào

CATS = ['Benign', 'Adware', 'Banking', 'SMS', 'Riskware']
found = {}
for f in sorted(os.listdir(ZIPS)):
    if not f.lower().endswith('.zip'):
        continue
    size = os.path.getsize(os.path.join(ZIPS, f))
    for c in CATS:
        if c.lower() in f.lower():       # khớp không phân biệt hoa thường
            found.setdefault(c, (f, size))
            break

for c in CATS:
    if c in found:
        f, s = found[c]
        print(f'  OK      {c:9s} {f}  ({s / 1e9:.2f} GB)')
    else:
        print(f'  THIEU   {c}')

missing = [c for c in CATS if c not in found]
if missing:
    print(f'\nCòn thiếu {len(missing)}/5: {missing}')
    print(f'Thư mục đã tạo sẵn: {ZIPS}')
    print('Tải tại https://www.unb.ca/cic/datasets/maldroid-2020.html'
          ' -> "Download the dataset" -> điền form,')
    print('rồi upload zip vào thư mục trên (hoặc dùng Drive trên máy cho nhanh).')
else:
    print('\nĐủ 5 category — chạy cell dưới.')

### Đường nhanh (tuỳ chọn) — tải thẳng bằng băng thông Colab

Link `browse.php?t=...` mà CIC đưa sau khi điền form **không tự nó cấp quyền**: quyền nằm ở **cookie phiên PHP** đặt lúc submit. Mở link đó ở máy khác hay ở Colab đều nhận:

```
Registration required. Please go back and register.
```

Nếu không muốn tải về máy rồi upload vài GB lên Drive, bạn có thể mượn cookie của chính phiên trình duyệt mình:

1. Trong trình duyệt đã điền form, mở **DevTools → Application → Cookies → cicresearch.ca**
2. Copy cặp `PHPSESSID=<giá trị>`
3. Dán vào cell dưới rồi chạy

Cookie chỉ sống một phiên và hết hạn khá nhanh — nếu gặp lại "Registration required" thì lấy cookie mới. **Đừng commit cookie vào repo**; cell dưới chỉ giữ nó trong biến môi trường của session này.

Nếu cách này không chạy (CIC có thể còn khoá theo IP), cứ quay lại cách chắc ăn: tải bằng trình duyệt rồi đưa vào `MyDrive/maldroid_zips/`.

In [ ]:
# Tuỳ chọn — chỉ chạy nếu muốn tải thẳng trên Colab thay vì upload từ máy.
import os

# Dán từ DevTools; giữ nguyên dạng 'PHPSESSID=...'
os.environ['CIC_COOKIE'] = 'PHPSESSID=THAY_BANG_COOKIE_CUA_BAN'

# Link browse.php CIC đưa sau khi điền form
BROWSE = 'https://cicresearch.ca/CICDataset/MalDroid-2020/browse.php?t=THAY_BANG_TOKEN'

# Thử liệt kê trước — nếu ra 5 category thì cookie còn hiệu lực
!python -m src.download --list --url-base "{BROWSE}"

In [ ]:
# Nếu cell trên liệt kê đủ 5 category thì tải luôn vào Drive.
# Chưa chắc chắn thì bỏ qua cell này và dùng cách upload thủ công.
!python -m src.download --url-base "{BROWSE}" --keep-zip

In [13]:
# Giải nén + sinh manifest.csv. Chạy lại được: category nào đã giải nén thì bỏ qua.
!python -m src.download --zip-dir "{ZIPS}"

# Nếu bạn đã tự giải nén sẵn thành apks/<category>/*.apk thì dùng dòng này:
# !python -m src.download --manifest-only

10:07:09 ERROR   download | Benign: khong tim thay zip trong /content/drive/MyDrive/maldroid_zips
10:07:10 ERROR   download | Adware: khong tim thay zip trong /content/drive/MyDrive/maldroid_zips
10:07:10 ERROR   download | Banking: khong tim thay zip trong /content/drive/MyDrive/maldroid_zips
10:07:10 ERROR   download | SMS: khong tim thay zip trong /content/drive/MyDrive/maldroid_zips
10:07:10 ERROR   download | Riskware: khong tim thay zip trong /content/drive/MyDrive/maldroid_zips
10:07:10 ERROR   download | Thieu category: ['Benign', 'Adware', 'Banking', 'SMS', 'Riskware']

Cach lay dataset (CIC yeu cau dien form, khong con tai an danh duoc):

  1. Mo https://www.unb.ca/cic/datasets/maldroid-2020.html
     bam "Download the dataset", dien form dang ky.
  2. Tai 5 file zip theo category ve mot thu muc - nen de tren Drive de khong
     phai lam lai moi session Colab.
  3. Chay lai voi thu muc do:

       python -m src.download --zip-dir /duong/dan/toi/thu/muc/zip

Neu da tu giai nen

In [14]:
# Chốt split. Sau lần chạy đầu, test_sha256.txt KHÔNG BAO GIỜ đổi nữa:
# các lần chạy sau sẽ tự đọc lại file đã chốt.
!python -m src.split

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/AndroiDetection1/src/split.py", line 260, in <module>
    main()
    ~~~~^^
  File "/content/AndroiDetection1/src/split.py", line 255, in main
    make_splits(args.manifest, args.n_train, args.n_val, args.n_test,
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                args.seed, args.force, not args.test_stratified)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/AndroiDetection1/src/split.py", line 117, in make_splits
    df = pd.read_csv(manifest_csv, dtype={"sha256": str})
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)

In [ ]:
# SMOKE TEST — PLAN mục 2: chạy Obfuscapk trên 1 APK với Rebuild.
# Nếu cell này fail thì DỪNG LẠI, mọi thứ sau đều vô nghĩa.
!python -m src.obfuscate --smoke-test

## Phase 2 — Trích feature tĩnh

Chỉ trích 7.500 APK trong split đã chốt, không phải cả 17k. Checkpoint mỗi 500 APK xuống Drive nên cell này resume được sau khi session chết.

In [ ]:
# Thử 20 APK trước để biết tốc độ thực tế và tỉ lệ fail
!python -m src.features.extract --tag clean --limit 20

In [ ]:
!python -m src.features.extract --tag clean
!cat $APKROB_WORK/features/extract_stats_clean.json

## Phase 3 — Baseline ML + eval sạch

In [ ]:
# Train model toàn bộ feature (rf/xgb/svm) + model chỉ-một-nhóm cho Bảng B
!python -m src.train

In [ ]:
!python -m src.evaluate --val --clean

In [ ]:
# Baseline đối chiếu trên CSV gốc của CIC — con số này so được trực tiếp với
# literature. Tải CSV từ trang CICMalDroid rồi trỏ đường dẫn vào đây.
CSV = '/content/drive/MyDrive/apk-robustness/feature_vectors_syscallsbinders_frequency_5_Cat.csv'
!test -f "$CSV" && python -m src.train --csv-baseline "$CSV" || echo 'Chưa có CSV — bỏ qua bước đối chiếu'

## Phase 4 — Obfuscation (9–10 giờ, chia 2 session)

Session A chạy T1–T3, session B chạy T4–T6. `obf_progress.json` nằm trên Drive nên chạy lại cell là resume, không làm lại từ đầu.

In [ ]:
# SESSION A
!python -m src.obfuscate --techniques T1_trivial T2_rename T3_string --workers 4

In [ ]:
# SESSION B
!python -m src.obfuscate --techniques T4_asset T5_cfg T6_reflection --workers 4

In [ ]:
!python -m src.obfuscate --report   # tỉ lệ APK hỏng theo từng kỹ thuật

In [ ]:
# Trích feature trên APK đã obfuscate.
# APK obfuscated nằm ở SCRATCH nên phải chạy cell này TRONG CÙNG session
# với cell obfuscate ở trên, trước khi Colab xoá disk.
for tech in ['T1_trivial','T2_rename','T3_string','T4_asset','T5_cfg','T6_reflection']:
    !python -m src.features.extract --tag {tech}

## Phase 5 — Ma trận kết quả

In [ ]:
!python -m src.evaluate --obf

In [ ]:
!python -m src.matrix

In [ ]:
from IPython.display import Markdown, display
import os
display(Markdown(open(os.path.join(os.environ['APKROB_WORK'], 'results', 'tables.md'), encoding='utf-8').read()))

## Pass sau — chỉ khi Bảng B đòi

`G5` (call graph) tốn 5–30s/APK, tức ~80% tổng thời gian Phase 2. Chỉ bật nếu Bảng B cho thấy G1–G4 không đủ tách bạch.

In [ ]:
# !python -m src.features.extract --tag clean --with-g5
# !python -m src.train --groups G1 G2 G3 G4 G5